In [56]:
import torch
import re
import pandas as pd

In [20]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Current available device: {device}")

Current available device: mps


## Import text

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

In [4]:
print("Total number of chars:", len(raw_text))
print(raw_text[:100])

Total number of chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


## Tokenizing text

### testing tokenizer

In [5]:
# input text
text = "Hello, world. Is this-a test?"

# tokenizing schema
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)

# tokenized text
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this-a', 'test', '?']


### Preprocessing (applying tokenizer on imported text)

In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(f"Token length of imported text: {len(preprocessed)}")
n = 30
print(f"\nFirst {n} tokens: {preprocessed[:n]}")

Token length of imported text: 4690

First 30 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into Token-IDs

In [64]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Size of words: {vocab_size}")
print(f"\nFirst {n} alphabetically ordered tokens {all_words[:n]}")

Size of words: 1130

First 30 alphabetically ordered tokens ['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed']


In [67]:
vocab = {token: ID for ID, token in enumerate(all_words)}

In [68]:
df = pd.DataFrame(vocab.items(), columns=["Token", "Token ID"])
df

,Token,Token ID
0,!,0
1,"""",1
2,',2
3,(,3
4,),4
...,...,...
1125,yet,1125
1126,you,1126
1127,younger,1127
1128,your,1128


## How to build a tokenizer

In [69]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # saves vocabulary as class attribute to access during Encode-Decode-methods
        self.str_to_int = vocab
        # creates inverse vocabulary (Token IDs to original text token)
        self.int_to_str = {integer:string for string,integer in vocab.items()}

    # transforms input text to Token IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ID_list = [self.str_to_int[string] for string in preprocessed]
        return ID_list

    # converts Token IDs back to text
    def decode(self, ID_list):
        text = " ".join([self.int_to_str[integer] for integer in ID_list])
        # removes space before defined punctuation marks
        text = re.sub(r'\s+([,.;:?!"()\'])', r'\1', text)
        return text

### Testing tokenizer (version 1)


In [72]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. 
"The last but one," she corrected herself--"but the other doesn't count, because he destroyed it."
"""

ID_list = tokenizer.encode(text)
print(f"Text to Token IDs:\n{ID_list}")
print(f"\n\nToken IDs to text:\n{tokenizer.decode(ID_list)}")

Text to Token IDs:
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7, 1, 93, 602, 239, 729, 5, 1, 876, 291, 542, 6, 1, 239, 988, 735, 356, 2, 970, 294, 5, 205, 533, 330, 585, 7, 1]


Token IDs to text:
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride." The last but one," she corrected herself --" but the other doesn' t count, because he destroyed it."
